# Notebook 08 — Tool Chaining

Claude calling tools in sequence, each step using the result of the previous one.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Agent loop with max_steps guard

In [ ]:
def run_agent(user_message, tools, dispatcher, max_steps=8):
    messages = [{"role": "user", "content": user_message}]
    print(f"User: {user_message}\n")
    for step in range(max_steps):
        r = client.messages.create(model=FAST_MODEL, max_tokens=512, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason == "end_turn":
            reply = next((b.text for b in r.content if hasattr(b, "text")), "")
            print(f"Final: {reply}")
            return reply
        if r.stop_reason == "tool_use":
            results = []
            for block in r.content:
                if block.type == "tool_use":
                    out = dispatcher(block.name, block.input)
                    print(f"  Step {step+1}: {block.name}({block.input}) -> {str(out)[:80]}")
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(out)})
            messages.append({"role": "user", "content": results})
    return "max_steps reached"


## Example: multi-step research + translate

In [ ]:
MOCK = {"search": "Paris is the capital of France with a population of 2.1 million in the city proper."}

CHAIN_TOOLS = [
    {"name": "search",    "description": "Search for info on a topic.", "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "translate", "description": "Translate text to a language.", "input_schema": {"type": "object", "properties": {"text": {"type": "string"}, "language": {"type": "string"}}, "required": ["text", "language"]}},
]

def dispatch(name, inputs):
    if name == "search":
        return MOCK.get("search", "No results found.")
    if name == "translate":
        r = client.messages.create(model=FAST_MODEL, max_tokens=200,
            messages=[{"role": "user", "content": f"Translate to {inputs['language']}: {inputs['text']}"}])
        return r.content[0].text
    return "unknown tool"

run_agent(
    "Search for info about Paris, then translate the result to German.",
    CHAIN_TOOLS, dispatch
)


## Exercise
Add a `summarize` tool and build a 3-step chain: search → summarize → translate.

In [ ]:
# Your code here
